# Module 06: Multiple Regression
*Statistics for Econometrics — An intermediate course for researchers*

This module extends simple regression to multiple predictors: omitted variable bias, ceteris paribus interpretation, the Frisch-Waugh theorem, adjusted R², multicollinearity, and joint hypothesis testing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    from statsmodels.stats.outliers_influence import variance_inflation_factor
except ImportError:
    !pip install statsmodels
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 6.1 Omitted Variable Bias

**Omitted Variable Bias (OVB)** occurs when a relevant variable is excluded from a regression model. This causes estimated coefficients to be biased and inconsistent.

### Conditions for OVB
Bias arises when TWO conditions hold simultaneously:
1. The omitted variable is a determinant of the outcome (correlated with Y)
2. The omitted variable is correlated with an included explanatory variable (correlated with X)

### Direction of Bias
The direction and magnitude of bias depends on the signs of these correlations:
$$\text{OVB} \approx \gamma \times \delta$$
where:
- γ = effect of omitted variable on X (cov(omitted, X))
- δ = direct effect of omitted variable on Y

If both positive or both negative → positive bias; if opposite signs → negative bias.

In [ ]:
# Demonstrate Omitted Variable Bias
# Population is our omitted variable (will exclude it, then include it)

n = 100
np.random.seed(42)

# Generate data
population = np.random.uniform(5000, 100000, n)

# Distance: larger communes tend to be closer (negative correlation with population)
distance = 50 - 0.0003 * population + np.random.normal(0, 8, n)

# Property value depends on both distance AND population
property_value = 200000 + 800*distance + 0.5*population + np.random.normal(0, 20000, n)

# Create dataframe
data_ovb = pd.DataFrame({
    'property_value': property_value,
    'distance': distance,
    'population': population
})

print("="*70)
print("SIMPLE REGRESSION (Distance only) - BIASED")
print("="*70)
model_simple = ols('property_value ~ distance', data=data_ovb).fit()
print(model_simple.summary())
beta_distance_simple = model_simple.params['distance']

print("\n" + "="*70)
print("MULTIPLE REGRESSION (Distance + Population) - UNBIASED")
print("="*70)
model_multiple = ols('property_value ~ distance + population', data=data_ovb).fit()
print(model_multiple.summary())
beta_distance_multiple = model_multiple.params['distance']

print("\n" + "="*70)
print("OMITTED VARIABLE BIAS ANALYSIS")
print("="*70)
ovb_magnitude = beta_distance_simple - beta_distance_multiple
print(f"β̂_distance (simple regression):    {beta_distance_simple:.2f}")
print(f"β̂_distance (multiple regression):  {beta_distance_multiple:.2f}")
print(f"Omitted Variable Bias (OVB):        {ovb_magnitude:.2f}")
print(f"\nThe bias is approximately γ̂ × δ̂:")

# Recover γ̂ and δ̂
# γ̂ = coefficient of population on distance
reg_gamma = ols('distance ~ population', data=data_ovb).fit()
gamma_hat = reg_gamma.params['population']

# δ̂ = coefficient of population on property_value (from full model)
delta_hat = model_multiple.params['population']

print(f"  γ̂ (population → distance):  {gamma_hat:.6f}")
print(f"  δ̂ (population → value):     {delta_hat:.2f}")
print(f"  γ̂ × δ̂:                       {gamma_hat * delta_hat:.2f}")
print(f"\nOVB theoretical ≈ {gamma_hat * delta_hat:.2f}, actual OVB = {ovb_magnitude:.2f}")

## 6.2 The Multiple Regression Model

The **multiple regression model** extends simple regression:
$$Y_i = \beta_0 + \beta_1 X_{1i} + \beta_2 X_{2i} + \cdots + \beta_k X_{ki} + \epsilon_i$$

Each coefficient β_j has a **ceteris paribus** interpretation: the effect of a one-unit increase in X_j, **holding all other predictors constant**.

In matrix notation:
$$\mathbf{Y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\epsilon}$$

The OLS estimator remains:
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{Y}$$

In [ ]:
# Expand dataset with additional predictors
np.random.seed(42)
n = 100

# Generate expanded dataset
population = np.random.uniform(5000, 100000, n)
distance = 50 - 0.0003 * population + np.random.normal(0, 8, n)
income_per_capita = 30000 + 0.08 * population + np.random.normal(0, 5000, n)
highway_access = np.random.binomial(1, 0.6, n)  # Binary: 1 if highway nearby, 0 otherwise
property_value = (200000 + 800*distance + 0.5*population + 0.15*income_per_capita 
                  + 50000*highway_access + np.random.normal(0, 20000, n))

data_full = pd.DataFrame({
    'property_value': property_value,
    'distance': distance,
    'population': population,
    'income_per_capita': income_per_capita,
    'highway_access': highway_access
})

# Fit full multiple regression model
model_full = ols('property_value ~ distance + population + income_per_capita + highway_access', 
                  data=data_full).fit()

print("FULL MULTIPLE REGRESSION MODEL")
print("="*70)
print(model_full.summary())

print("\n" + "="*70)
print("COEFFICIENT INTERPRETATION (Ceteris Paribus)")
print("="*70)
for var, coef in model_full.params.items():
    if var == 'Intercept':
        print(f"Intercept: £{coef:,.0f}")
    elif var == 'distance':
        print(f"Distance: £{coef:,.0f} per km increase (holding other variables constant)")
    elif var == 'population':
        print(f"Population: £{coef:.2f} per person increase (holding other variables constant)")
    elif var == 'income_per_capita':
        print(f"Income per capita: £{coef:.3f} per £1 increase (holding other variables constant)")
    elif var == 'highway_access':
        print(f"Highway access: £{coef:,.0f} premium for having highway access (holding other variables constant)")

# Predictions vs actual
predictions = model_full.fittedvalues
residuals = model_full.resid

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(predictions, data_full['property_value'], alpha=0.6, s=50)
ax.plot([predictions.min(), predictions.max()], [predictions.min(), predictions.max()], 
        'r--', lw=2, label='Perfect fit')
ax.set_xlabel('Fitted values (£)', fontsize=11)
ax.set_ylabel('Actual property value (£)', fontsize=11)
ax.set_title('Multiple Regression: Fitted vs Actual Values', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nR² = {model_full.rsquared:.4f}")
print(f"RMSE = £{np.sqrt(np.mean(residuals**2)):,.0f}")

## 6.3 The Frisch-Waugh-Lovell Theorem

The **Frisch-Waugh-Lovell (FWL) Theorem** reveals the mechanical meaning of "controlling for" a variable.

### What it says
The coefficient on X₁ in the multiple regression Y ~ X₁ + X₂ equals the coefficient from:
1. Regress X₁ on X₂, save residuals (residual variation in X₁ after removing X₂'s effect)
2. Regress Y on X₂, save residuals (residual variation in Y after removing X₂'s effect)
3. Regress residual_Y on residual_X₁

### Interpretation
The coefficient represents the relationship between X₁ and Y **after partialling out the effect of X₂**. In other words, it captures the correlation between X₁ and Y that remains after both have been "cleaned" of their linear relationship with X₂.

In [ ]:
# Frisch-Waugh-Lovell Theorem Demonstration
# We'll focus on: property_value ~ distance + population

print("="*70)
print("FRISCH-WAUGH-LOVELL THEOREM DEMONSTRATION")
print("="*70)

print("\nStep 1: Regress distance on population, save residuals")
reg_x1_on_x2 = ols('distance ~ population', data=data_ovb).fit()
residuals_distance = reg_x1_on_x2.resid
print(f"Residual std dev: {residuals_distance.std():.3f}")

print("\nStep 2: Regress property_value on population, save residuals")
reg_y_on_x2 = ols('property_value ~ population', data=data_ovb).fit()
residuals_value = reg_y_on_x2.resid
print(f"Residual std dev: {residuals_value.std():.2f}")

print("\nStep 3: Regress residual_value on residual_distance")
data_residuals = pd.DataFrame({
    'residuals_value': residuals_value,
    'residuals_distance': residuals_distance
})
reg_fwl = ols('residuals_value ~ residuals_distance', data=data_residuals).fit()
beta_fwl = reg_fwl.params['residuals_distance']
print(reg_fwl.summary())

print("\n" + "="*70)
print("COMPARISON WITH FULL MULTIPLE REGRESSION")
print("="*70)
beta_multiple = model_multiple.params['distance']
print(f"Coefficient from multiple regression:     {beta_multiple:.4f}")
print(f"Coefficient from FWL approach:            {beta_fwl:.4f}")
print(f"\nDifference: {abs(beta_multiple - beta_fwl):.10f} (numerical precision)")
print("\n✓ They are identical! This is the Frisch-Waugh-Lovell theorem.")

# Visualise the partialling out
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw relationship
axes[0].scatter(data_ovb['distance'], data_ovb['property_value'], alpha=0.6, s=50)
z = np.polyfit(data_ovb['distance'], data_ovb['property_value'], 1)
p = np.poly1d(z)
x_line = np.linspace(data_ovb['distance'].min(), data_ovb['distance'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', lw=2, label=f'β = {z[0]:.0f}')
axes[0].set_xlabel('Distance (km)', fontsize=10)
axes[0].set_ylabel('Property value (£)', fontsize=10)
axes[0].set_title('Raw relationship (biased)', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# After partialling out population
axes[1].scatter(residuals_distance, residuals_value, alpha=0.6, s=50, color='orange')
z2 = np.polyfit(residuals_distance, residuals_value, 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(residuals_distance.min(), residuals_distance.max(), 100)
axes[1].plot(x_line2, p2(x_line2), 'r--', lw=2, label=f'β = {z2[0]:.0f}')
axes[1].set_xlabel('Distance | Population residuals', fontsize=10)
axes[1].set_ylabel('Property value | Population residuals', fontsize=10)
axes[1].set_title('After partialling out population (unbiased)', fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Coefficients comparison
methods = ['Simple\n(biased)', 'Multiple\n(unbiased)', 'FWL\n(unbiased)']
coefs = [beta_distance_simple, beta_distance_multiple, beta_fwl]
colors = ['lightcoral', 'lightgreen', 'lightgreen']
axes[2].bar(methods, coefs, color=colors, edgecolor='black', linewidth=1.5)
axes[2].set_ylabel('Coefficient value (£/km)', fontsize=10)
axes[2].set_title('Coefficient comparison', fontsize=11, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(coefs):
    axes[2].text(i, v + 10, f'{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6.4 Adjusted R² and Model Comparison

### The Problem with R²
R² always increases (or stays the same) when you add variables, even if those variables are **irrelevant noise**. This is because adding any variable can only reduce RSS (residual sum of squares) or keep it the same.

### Adjusted R²
**Adjusted R²** penalises model complexity by adjusting for the number of predictors:
$$\bar{R}^2 = 1 - \frac{\text{RSS}/(n-k-1)}{\text{TSS}/(n-1)} = 1 - (1-R^2) \frac{n-1}{n-k-1}$$

where k = number of predictors (excluding intercept).

- Adjusted R² **can decrease** when irrelevant variables are added
- It rewards parsimonious models
- Useful for model selection alongside **AIC** and **BIC**

In [ ]:
# Demonstrate R² vs Adjusted R² as variables are added
np.random.seed(42)
n = 100

# Use our expanded dataset
# Add random noise variables (irrelevant)
for i in range(3):
    data_full[f'noise_{i}'] = np.random.normal(0, 1, n)

# Fit models with increasing complexity
model_list = []
model_names = []
r_squared = []
adj_r_squared = []
aic_vals = []
bic_vals = []

# Model 1: Only distance
m1 = ols('property_value ~ distance', data=data_full).fit()
model_list.append(m1)
model_names.append('distance')
r_squared.append(m1.rsquared)
adj_r_squared.append(m1.rsquared_adj)
aic_vals.append(m1.aic)
bic_vals.append(m1.bic)

# Model 2: distance + population
m2 = ols('property_value ~ distance + population', data=data_full).fit()
model_list.append(m2)
model_names.append('distance + population')
r_squared.append(m2.rsquared)
adj_r_squared.append(m2.rsquared_adj)
aic_vals.append(m2.aic)
bic_vals.append(m2.bic)

# Model 3: distance + population + income_per_capita
m3 = ols('property_value ~ distance + population + income_per_capita', data=data_full).fit()
model_list.append(m3)
model_names.append('+ income_per_capita')
r_squared.append(m3.rsquared)
adj_r_squared.append(m3.rsquared_adj)
aic_vals.append(m3.aic)
bic_vals.append(m3.bic)

# Model 4: add highway_access
m4 = ols('property_value ~ distance + population + income_per_capita + highway_access', 
         data=data_full).fit()
model_list.append(m4)
model_names.append('+ highway_access')
r_squared.append(m4.rsquared)
adj_r_squared.append(m4.rsquared_adj)
aic_vals.append(m4.aic)
bic_vals.append(m4.bic)

# Model 5: add first noise variable
m5 = ols('property_value ~ distance + population + income_per_capita + highway_access + noise_0', 
         data=data_full).fit()
model_list.append(m5)
model_names.append('+ noise_0')
r_squared.append(m5.rsquared)
adj_r_squared.append(m5.rsquared_adj)
aic_vals.append(m5.aic)
bic_vals.append(m5.bic)

# Model 6: add second noise variable
m6 = ols('property_value ~ distance + population + income_per_capita + highway_access + noise_0 + noise_1', 
         data=data_full).fit()
model_list.append(m6)
model_names.append('+ noise_1')
r_squared.append(m6.rsquared)
adj_r_squared.append(m6.rsquared_adj)
aic_vals.append(m6.aic)
bic_vals.append(m6.bic)

# Model 7: add third noise variable
m7 = ols('property_value ~ distance + population + income_per_capita + highway_access + noise_0 + noise_1 + noise_2', 
         data=data_full).fit()
model_list.append(m7)
model_names.append('+ noise_2')
r_squared.append(m7.rsquared)
adj_r_squared.append(m7.rsquared_adj)
aic_vals.append(m7.aic)
bic_vals.append(m7.bic)

# Create comparison table
comparison_df = pd.DataFrame({
    'Model': model_names,
    'R²': r_squared,
    'Adjusted R²': adj_r_squared,
    'AIC': aic_vals,
    'BIC': bic_vals
})

print("MODEL COMPARISON TABLE")
print("="*80)
print(comparison_df.to_string(index=False))
print("\nObservations:")
print("  • R² increases monotonically (always increases with added variables)")
print("  • Adjusted R² decreases after adding noise variables (penalises overfitting)")
print("  • AIC and BIC also penalise model complexity")
print("  • Best model by Adjusted R², AIC, BIC: Model 4 (highway_access)")

# Visualise R² vs Adjusted R²
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(model_names))
width = 0.35

ax1.bar(x_pos - width/2, r_squared, width, label='R²', alpha=0.8)
ax1.bar(x_pos + width/2, adj_r_squared, width, label='Adjusted R²', alpha=0.8)
ax1.axvline(x=3.5, color='red', linestyle='--', linewidth=2, label='Noise vars added →')
ax1.set_ylabel('R² value', fontsize=11)
ax1.set_title('R² vs Adjusted R²: Model Comparison', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'M{i+1}' for i in range(len(model_names))], fontsize=10)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim([0, 1])

# AIC/BIC (lower is better)
ax2.plot(range(len(model_names)), aic_vals, marker='o', linewidth=2, markersize=8, label='AIC')
ax2.plot(range(len(model_names)), bic_vals, marker='s', linewidth=2, markersize=8, label='BIC')
ax2.axvline(x=3, color='red', linestyle='--', linewidth=2, label='Noise vars added →')
ax2.set_ylabel('Information Criterion (lower is better)', fontsize=11)
ax2.set_xlabel('Model', fontsize=11)
ax2.set_title('AIC and BIC: Penalising Complexity', fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(model_names)))
ax2.set_xticklabels([f'M{i+1}' for i in range(len(model_names))], fontsize=10)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.5 Multicollinearity

**Multicollinearity** occurs when two or more explanatory variables are highly correlated with each other.

### Consequences
- **Individual coefficients** become imprecise (large standard errors)
- **t-statistics shrink**, making individual variables appear insignificant
- **Joint F-test** may remain significant (the variables collectively predict Y)
- **Coefficient estimates** become unstable: small changes in data cause large coefficient changes
- **R² unaffected**: the overall model fit is not harmed

### Detecting Multicollinearity
1. **Correlation matrix**: look for correlations > 0.8 or 0.9
2. **Variance Inflation Factor (VIF)**: 
   - VIF_j = 1/(1 - R²_j) where R²_j is from regressing X_j on other X variables
   - VIF > 5 or 10 suggests problematic collinearity
   - VIF = 1 means no collinearity

In [ ]:
# Create multicollinear variables
np.random.seed(42)
n = 100

# distance is measured in km
distance_km = np.linspace(10, 50, n) + np.random.normal(0, 2, n)

# travel_time is strongly correlated with distance_km
# travel_time ≈ 1.5 * distance + noise (high correlation)
travel_time_min = 15 + 1.5 * distance_km + np.random.normal(0, 3, n)

# Independent variable
income = np.random.uniform(25000, 75000, n)

# Outcome: distance matters for property value
property_value_multi = 250000 - 2000*distance_km + 0.1*income + np.random.normal(0, 15000, n)

data_multi = pd.DataFrame({
    'property_value': property_value_multi,
    'distance_km': distance_km,
    'travel_time_min': travel_time_min,
    'income': income
})

# Check correlation
corr_matrix = data_multi[['distance_km', 'travel_time_min', 'income', 'property_value']].corr()
print("CORRELATION MATRIX")
print("="*50)
print(corr_matrix.round(3))
print(f"\nCorrelation between distance_km and travel_time_min: {corr_matrix.loc['distance_km', 'travel_time_min']:.3f}")
print("→ Very high collinearity!")

# Fit model with multicollinear variables
print("\n" + "="*70)
print("MODEL WITH MULTICOLLINEAR VARIABLES")
print("="*70)
model_multi = ols('property_value ~ distance_km + travel_time_min + income', data=data_multi).fit()
print(model_multi.summary())

print("\nNote: Both distance_km and travel_time_min are individually insignificant (t ~ 0)")
print("BUT they are conceptually important and jointly significant (F-test would be significant).")

# Calculate VIF
print("\n" + "="*70)
print("VARIANCE INFLATION FACTOR (VIF)")
print("="*70)

# Need to add constant manually for VIF calculation
X_with_const = sm.add_constant(data_multi[['distance_km', 'travel_time_min', 'income']])

vif_data = pd.DataFrame()
vif_data['Variable'] = ['distance_km', 'travel_time_min', 'income']
vif_data['VIF'] = [variance_inflation_factor(X_with_const.values, i) 
                    for i in range(1, X_with_const.shape[1])]

print(vif_data.to_string(index=False))
print("\nInterpretation:")
print("  • VIF > 5: problematic multicollinearity")
print(f"  • distance_km and travel_time_min both have very high VIF (> 50)")
print("  • These variables explain nearly all each other's variance")

# Compare with model without collinear variable
print("\n" + "="*70)
print("MODEL WITHOUT ONE COLLINEAR VARIABLE (distance_km only)")
print("="*70)
model_no_multi = ols('property_value ~ distance_km + income', data=data_multi).fit()
print(model_no_multi.summary())
print("\nNow distance_km is statistically significant and coefficient is stable!")

# Visualise collinearity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[0], cbar_kws={'label': 'Correlation'})
axes[0].set_title('Correlation Matrix: Multicollinearity Detection', fontsize=12, fontweight='bold')

# VIF visualisation
vif_sorted = vif_data.sort_values('VIF')
colors_vif = ['lightcoral' if x > 5 else 'lightgreen' for x in vif_sorted['VIF']]
axes[1].barh(vif_sorted['Variable'], vif_sorted['VIF'], color=colors_vif, edgecolor='black')
axes[1].axvline(x=5, color='red', linestyle='--', linewidth=2, label='VIF = 5 threshold')
axes[1].set_xlabel('VIF', fontsize=11)
axes[1].set_title('Variance Inflation Factor', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='x')

for i, v in enumerate(vif_sorted['VIF']):
    axes[1].text(v + 2, i, f'{v:.1f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Bootstrap coefficient stability
print("\n" + "="*70)
print("COEFFICIENT STABILITY: Bootstrap Estimates")
print("="*70)

n_bootstrap = 1000
boots_coef_with = []
boots_coef_without = []

for b in range(n_bootstrap):
    indices = np.random.choice(n, n, replace=True)
    boot_data = data_multi.iloc[indices]
    
    # With collinear variable
    m_with = ols('property_value ~ distance_km + travel_time_min + income', data=boot_data).fit()
    boots_coef_with.append(m_with.params['distance_km'])
    
    # Without collinear variable
    m_without = ols('property_value ~ distance_km + income', data=boot_data).fit()
    boots_coef_without.append(m_without.params['distance_km'])

print(f"Coefficient on distance_km:")
print(f"  With collinear variable:    mean = {np.mean(boots_coef_with):.0f}, std = {np.std(boots_coef_with):.0f}")
print(f"  Without collinear variable: mean = {np.mean(boots_coef_without):.0f}, std = {np.std(boots_coef_without):.0f}")
print(f"\nThe collinear variable increases coefficient instability!")

## 6.6 Variable Selection Strategy

### A Framework for Choosing Variables

**Start with theory.** Ask: What variables should logically affect the outcome?

1. **Confounders**: Variables that affect both X and Y. These cause bias if omitted.
   - Example: neighbourhood quality affects both distance and property value
   - **Action**: Include confounders to get unbiased estimates

2. **Mediators**: Variables through which X affects Y. Y ← X → Mediator → Y
   - Example: distance → commute time → property value
   - **Action**: Exclude mediators if you want total effect; include to estimate direct effect

3. **Colliders**: Variables affected by X, that don't affect Y directly.
   - **Action**: Never condition on colliders (creates bias!)

4. **Irrelevant variables**: No causal relationship to Y.
   - **Action**: Exclude to keep model simple and avoid multicollinearity

### Bad Controls
A variable is a **"bad control"** if:
- It is affected by the treatment/regressor (e.g., a mediator)
- Conditioning on it attenuates the coefficient we care about
- We risk misinterpreting a direct effect as absent

In [ ]:
# Bad Control Demonstration
# Causal chain: distance → commute_time → property_value

np.random.seed(42)
n = 100

# Distance from city centre
distance = np.linspace(5, 50, n) + np.random.normal(0, 2, n)

# Commute time increases with distance (mediation mechanism)
commute_time = 10 + 2.5 * distance + np.random.normal(0, 5, n)

# Property value depends on distance AND commute time
# But commute time is HOW distance affects property value (a mediator)
property_value_bad = 300000 - 3000*distance + np.random.normal(0, 20000, n)

data_bad_control = pd.DataFrame({
    'property_value': property_value_bad,
    'distance': distance,
    'commute_time': commute_time
})

print("="*70)
print("BAD CONTROL EXAMPLE: Causal chain: distance → commute_time → value")
print("="*70)

print("\nModel 1: Distance only (TRUE TOTAL EFFECT)")
print("-" * 70)
m1_bad = ols('property_value ~ distance', data=data_bad_control).fit()
print(m1_bad.summary())
beta_distance_alone = m1_bad.params['distance']
print(f"\nInterpretation: A 1 km increase in distance reduces property value")
print(f"               by £{abs(beta_distance_alone):.0f} (through commute time and other channels).")

print("\n" + "="*70)
print("Model 2: Distance + Commute time (CONTROLLING FOR MEDIATOR - BAD!)")
print("="*70)
m2_bad = ols('property_value ~ distance + commute_time', data=data_bad_control).fit()
print(m2_bad.summary())
beta_distance_controlled = m2_bad.params['distance']
print(f"\nInterpretation: Controlling for commute time, distance coefficient ≈ {beta_distance_controlled:.0f}")
print(f"               (attenuated from {beta_distance_alone:.0f})")
print(f"\n⚠️  This is a BAD CONTROL:")
print(f"   • We control for commute_time (a mediator)")
print(f"   • This blocks the causal path: distance → commute_time → value")
print(f"   • We underestimate the total effect of distance")
print(f"   • We might incorrectly conclude distance doesn't matter")

# Visualise the attenuation
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Distance → commute_time
axes[0, 0].scatter(distance, commute_time, alpha=0.6, s=50, color='steelblue')
z1 = np.polyfit(distance, commute_time, 1)
p1 = np.poly1d(z1)
x_line1 = np.linspace(distance.min(), distance.max(), 100)
axes[0, 0].plot(x_line1, p1(x_line1), 'r--', lw=2)
axes[0, 0].set_xlabel('Distance (km)', fontsize=10)
axes[0, 0].set_ylabel('Commute time (min)', fontsize=10)
axes[0, 0].set_title('First step: distance → commute_time', fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# 2. Commute_time → property_value
axes[0, 1].scatter(commute_time, property_value_bad, alpha=0.6, s=50, color='coral')
z2 = np.polyfit(commute_time, property_value_bad, 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(commute_time.min(), commute_time.max(), 100)
axes[0, 1].plot(x_line2, p2(x_line2), 'r--', lw=2)
axes[0, 1].set_xlabel('Commute time (min)', fontsize=10)
axes[0, 1].set_ylabel('Property value (£)', fontsize=10)
axes[0, 1].set_title('Second step: commute_time → value', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Raw effect: distance → property_value (total effect)
axes[1, 0].scatter(distance, property_value_bad, alpha=0.6, s=50, color='green')
z3 = np.polyfit(distance, property_value_bad, 1)
p3 = np.poly1d(z3)
x_line3 = np.linspace(distance.min(), distance.max(), 100)
axes[1, 0].plot(x_line3, p3(x_line3), 'r--', lw=2, label=f'β = {z3[0]:.0f}')
axes[1, 0].set_xlabel('Distance (km)', fontsize=10)
axes[1, 0].set_ylabel('Property value (£)', fontsize=10)
axes[1, 0].set_title('Total effect (correct): distance → value', fontsize=11, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# 4. Coefficient comparison
models = ['Without\ncommute_time\n(Correct)', 'With\ncommute_time\n(Bad control)']
coefs = [beta_distance_alone, beta_distance_controlled]
colors = ['lightgreen', 'lightcoral']
axes[1, 1].bar(models, coefs, color=colors, edgecolor='black', linewidth=1.5, alpha=0.7)
axes[1, 1].set_ylabel('Coefficient on distance', fontsize=10)
axes[1, 1].set_title('Effect attenuation from bad control', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(coefs):
    axes[1, 1].text(i, v - 200, f'{v:.0f}', ha='center', va='top', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY TAKEAWAY")
print("="*70)
print("Control for confounders (variables that cause X and Y).")
print("Do NOT control for mediators (variables that X causes, that then affect Y).")

## 6.7 Joint Hypothesis Testing (F-test)

### Testing Multiple Coefficients Simultaneously
The **F-test** for nested models tests whether a group of variables jointly significantly improve the model.

**Null hypothesis (H₀)**: β₁ = β₂ = ... = β_q = 0 (the q variables have no effect)

**F-statistic**:
$$F = \frac{(\text{RSS}_{\text{restricted}} - \text{RSS}_{\text{unrestricted}})/q}{\text{RSS}_{\text{unrestricted}}/(n-k-1)}$$

where:
- RSS_restricted = residual sum of squares from model WITHOUT the q variables
- RSS_unrestricted = residual sum of squares from model WITH the q variables
- q = number of restrictions (variables being tested)
- n = sample size, k = total number of predictors in unrestricted model

Under H₀, F ~ F(q, n-k-1)

In [ ]:
# F-test for Joint Hypothesis Testing
# Using our property value data

print("="*70)
print("JOINT HYPOTHESIS TESTING: F-TEST")
print("="*70)

print("\nResearch Question: Do population and income jointly affect property values?")
print("\nH₀: β_population = β_income_per_capita = 0")
print("H₁: At least one of these coefficients ≠ 0")

print("\n" + "-"*70)
print("Restricted Model (excludes population and income)")
print("-"*70)
model_restricted = ols('property_value ~ distance + highway_access', data=data_full).fit()
print(model_restricted.summary())
rss_restricted = np.sum(model_restricted.resid**2)
print(f"\nRSS (restricted): {rss_restricted:.2e}")

print("\n" + "-"*70)
print("Unrestricted Model (includes population and income)")
print("-"*70)
# Using our full model
print(model_full.summary())
rss_unrestricted = np.sum(model_full.resid**2)
print(f"\nRSS (unrestricted): {rss_unrestricted:.2e}")

# Calculate F-statistic manually
q = 2  # number of restrictions (population and income)
n = len(data_full)
k = 4  # number of predictors in unrestricted model

f_stat_manual = ((rss_restricted - rss_unrestricted) / q) / (rss_unrestricted / (n - k - 1))
p_value_f = 1 - stats.f.cdf(f_stat_manual, q, n - k - 1)

print("\n" + "="*70)
print("F-TEST CALCULATION")
print("="*70)
print(f"RSS (restricted) - RSS (unrestricted) = {rss_restricted - rss_unrestricted:.2e}")
print(f"Number of restrictions (q) = {q}")
print(f"Numerator: ({rss_restricted:.2e} - {rss_unrestricted:.2e}) / {q}")
print(f"         = {(rss_restricted - rss_unrestricted) / q:.2e}")
print(f"\nDenominator: {rss_unrestricted:.2e} / ({n} - {k} - 1)")
print(f"           = {rss_unrestricted:.2e} / {n - k - 1}")
print(f"           = {rss_unrestricted / (n - k - 1):.2e}")
print(f"\nF-statistic = {(rss_restricted - rss_unrestricted) / q:.2e} / {rss_unrestricted / (n - k - 1):.2e}")
print(f"            = {f_stat_manual:.4f}")
print(f"\nDegrees of freedom: F({q}, {n - k - 1})")
print(f"P-value: {p_value_f:.6f}")

if p_value_f < 0.05:
    print(f"\n✓ REJECT H₀ at α=0.05 level")
    print(f"  The variables 'population' and 'income_per_capita' jointly")
    print(f"  have a statistically significant effect on property values.")
else:
    print(f"\n✗ FAIL TO REJECT H₀ at α=0.05 level")
    print(f"  Insufficient evidence that population and income jointly affect values.")

# Verify with statsmodels (using anova_lm)
print("\n" + "="*70)
print("VERIFICATION using statsmodels anova_lm")
print("="*70)

from statsmodels.stats.anova import anova_lm
anova_table = anova_lm(model_restricted, model_full)
print(anova_table)
print(f"\nF-statistic (from anova_lm): {anova_table.loc[1, 'F']:.4f}")
print(f"P-value (from anova_lm): {anova_table.loc[1, 'Pr(>F)']:.6f}")

# Overall F-test
print("\n" + "="*70)
print("OVERALL F-TEST: Are all predictors jointly significant?")
print("="*70)
print(f"\nFrom full model summary:")
print(f"F-statistic: {model_full.fvalue:.4f}")
print(f"Prob (F-statistic): {model_full.f_pvalue:.6e}")
print(f"\nInterpretation: Are all {k} predictors (as a group) significant?")
if model_full.f_pvalue < 0.05:
    print(f"✓ YES: The model as a whole is statistically significant.")
else:
    print(f"✗ NO: The model explains no more than a constant.")

## Exercises

### Exercise 1: Omitted Variable Bias
Two researchers estimate the effect of class size on test scores:
- **Researcher A** regresses test scores on class size only, finding β̂ = +5
- **Researcher B** includes school funding, finding β̂ = -3

**Tasks:**
1. Explain which estimate is biased and why
2. What must be true about the relationships between (class size, funding, test scores) for OVB to exist?
3. Sketch the direction and sign of OVB

### Exercise 2: Interpreting Coefficients
A regression of ln(salary) on years of education and experience (years post-graduation) yields:
- β_education = 0.08
- β_experience = 0.03

**Tasks:**
1. Interpret each coefficient in context
2. What does "ceteris paribus" mean here?
3. If education and experience are correlated, how would excluding experience change the education coefficient?

### Exercise 3: Multicollinearity
A researcher fits the model: Y ~ X₁ + X₂ + X₃

VIF values: X₁=1.2, X₂=8.5, X₃=9.1. R²=0.78, Adjusted R²=0.76.

**Tasks:**
1. Identify which variables have problematic collinearity
2. What would you expect to see in the t-statistics for these variables?
3. Suggest remedies for the multicollinearity problem

### Exercise 4: Model Selection and F-tests
Three nested models:
- Model A: Y ~ X₁
- Model B: Y ~ X₁ + X₂ + X₃ (RSS = 2500)
- Model C: Y ~ X₁ + X₂ + X₃ + X₄ + X₅ (RSS = 2400)

Sample size n=100.

**Tasks:**
1. Test whether X₂ and X₃ jointly add explanatory power (B vs A)
2. Test whether X₄ and X₅ jointly add explanatory power (C vs B)
3. Which model would you prefer and why?

### Exercise Starter Code

Work through these exercises in the cells below. Show your reasoning clearly and interpret results in context.

In [ ]:
# EXERCISE 1: Omitted Variable Bias
# Fill in your analysis below

print("Exercise 1: Omitted Variable Bias")
print("="*70)
print("\nScenario: Test scores ~ class size")
print("  Researcher A (simple): β̂ = +5 (no controls)")
print("  Researcher B (multiple): β̂ = -3 (controls for funding)")
print("\nYour analysis:")
print("  1. Which coefficient is biased and why?")
print("     → Answer: ")
print("\n  2. What must be true for OVB?")
print("     → Answer: ")
print("\n  3. What is the sign and direction of bias?")
print("     → Answer: ")
print("\nOVB = (correlation of funding with class size) × (effect of funding on scores)")
print("     = (expected relationship) × (expected relationship)")
print("     = ...")

In [ ]:
# EXERCISE 2: Interpreting Coefficients
print("Exercise 2: Coefficient Interpretation")
print("="*70)
print("\nModel: ln(salary) = β₀ + β_education × education + β_experience × experience + ε")
print("\nResults:")
print("  β_education = 0.08")
print("  β_experience = 0.03")
print("\n1. Interpretation of education coefficient:")
print("   → ")
print("\n2. Interpretation of experience coefficient:")
print("   → ")
print("\n3. What does ceteris paribus mean in this context?")
print("   → ")

In [ ]:
# EXERCISE 3: Multicollinearity Detection
import pandas as pd

print("Exercise 3: Multicollinearity")
print("="*70)

vif_values = {'X1': 1.2, 'X2': 8.5, 'X3': 9.1}
r_squared = 0.78
adj_r_squared = 0.76

vif_df = pd.DataFrame(list(vif_values.items()), columns=['Variable', 'VIF'])
print("\nVIF Values:")
print(vif_df.to_string(index=False))
print(f"\nModel fit: R² = {r_squared:.2f}, Adjusted R² = {adj_r_squared:.2f}")

print("\n1. Which variables have problematic multicollinearity?")
print("   → ")

print("\n2. What would you expect in the t-statistics?")
print("   → ")

print("\n3. What remedies would you suggest?")
print("   → ")

In [ ]:
# EXERCISE 4: F-tests for Model Comparison
from scipy import stats

print("Exercise 4: F-test for Nested Models")
print("="*70)

# Given data
n = 100
rss_model_a = None  # Unknown
rss_model_b = 2500
rss_model_c = 2400

print("\nModel specifications:")
print("  Model A: Y ~ X₁")
print("  Model B: Y ~ X₁ + X₂ + X₃ (RSS = 2500)")
print("  Model C: Y ~ X₁ + X₂ + X₃ + X₄ + X₅ (RSS = 2400)")
print(f"\n  Sample size: n = {n}")

print("\n1. Test: Do X₂ and X₃ jointly improve over Model A?")
print("   Problem: We don't have RSS for Model A.")
print("   What information would you need?")
print("   → ")

print("\n2. Test: Do X₄ and X₅ jointly improve over Model B?")
q = 2  # X₄ and X₅
k_unrestricted = 5  # X₁, X₂, X₃, X₄, X₅
f_stat = ((rss_model_b - rss_model_c) / q) / (rss_model_c / (n - k_unrestricted - 1))
p_value = 1 - stats.f.cdf(f_stat, q, n - k_unrestricted - 1)

print(f"   F-statistic = ((2500 - 2400) / 2) / (2400 / (100 - 5 - 1))")
print(f"               = (100/2) / (2400/94)")
print(f"               = 50 / 25.53")
print(f"               = {f_stat:.4f}")
print(f"   P-value = {p_value:.4f}")
print(f"\n   Conclusion:")
print(f"   → ")

print("\n3. Which model would you prefer and why?")
print("   → ")